In [2]:
import numpy as np
import pandas as pd

1. Tạo dữ liệu mô phỏng AQI

In [ ]:
np.random.seed(42)
n = 1000  
dates = pd.date_range("2021-01-01", periods=n)
t = np.arange(n)

# Seasonal rõ ràng, ít noise hơn để LSTM học được
seasonal = 30 * np.sin(2 * np.pi * t / 365 + np.pi)
trend    = np.linspace(0, 15, n)
noise    = np.random.normal(0, 8, n)   # noise nhỏ hơn, bỏ spike
aqi      = 100 + seasonal + trend + noise
aqi      = np.clip(aqi, 30, 220)

pm25     = aqi * 0.4 + np.random.normal(0, 3, n)
pm25     = np.clip(pm25, 10, 150)
temp     = 23 + 8*np.sin(2*np.pi*t/365 - np.pi/2) + np.random.normal(0, 1.5, n)
humidity = 75 - 10*np.sin(2*np.pi*t/365 - np.pi/2) + np.random.normal(0, 3, n)
humidity = np.clip(humidity, 40, 100)

df_aqi = pd.DataFrame({
    "date": dates, "AQI": aqi,
    "PM25": pm25, "temp": temp, "humidity": humidity
})
df_aqi.to_csv("data_set/aqi_hanoi_2023.csv", index=False)
print(df_aqi.describe().round(2))

                      date      AQI     PM25     temp  humidity
count                 1000  1000.00  1000.00  1000.00   1000.00
mean   2022-05-15 12:00:00   105.78    42.53    23.47     74.37
min    2021-01-01 00:00:00    51.47    18.74    11.14     57.63
25%    2021-09-07 18:00:00    85.11    34.33    17.91     67.97
50%    2022-05-15 12:00:00   104.66    42.11    24.09     73.99
75%    2023-01-20 06:00:00   126.44    50.38    28.81     80.90
max    2023-09-27 00:00:00   159.38    66.08    36.53     94.44
std                    NaN    23.55     9.81     5.85      7.66


2. Tạo dữ liệu mô phỏng chất lượng nước sông Hồng (WQI)

In [ ]:
np.random.seed(42)
n = 365
dates = pd.date_range('2022-01-01', periods=n)
wqi = 70 + 15*np.sin(np.linspace(0, 4*np.pi, n)) + np.random.normal(0, 5, n)
wqi = np.clip(wqi, 20, 100)

df_water = pd.DataFrame({
    'timestamp':    dates,
    'WQI':          wqi,
    'pH':           np.random.normal(7.2, 0.3, n),
    'DO':           np.random.normal(6.5, 0.8, n),  
    'BOD':          np.random.normal(8, 1.5, n),     
    'turbidity':    np.random.normal(15, 3, n),
    'conductivity': np.random.normal(300, 30, n)
})

anomaly_idx = [145, 165, 178, 195, 203, 307]
df_water.loc[anomaly_idx, 'WQI']  = np.random.uniform(20, 35, 6)
df_water.loc[anomaly_idx, 'pH']   = np.random.uniform(4.5, 5.5, 6)
df_water.loc[anomaly_idx, 'DO']   = np.random.uniform(1.5, 2.5, 6)
df_water.loc[anomaly_idx, 'BOD']  = np.random.uniform(25, 35, 6)

df_water.to_csv("data_set/water_quality_songhong_2022.csv", index=False)
print(df_water.iloc[anomaly_idx][['timestamp','WQI','pH','DO','BOD']])

     timestamp        WQI        pH        DO        BOD
145 2022-05-26  34.409405  5.069540  2.481178  30.214596
165 2022-06-15  22.953886  4.759542  1.690108  34.773079
178 2022-06-28  34.271447  4.936996  2.292595  32.573102
195 2022-07-15  34.922289  5.093561  2.407899  26.616714
203 2022-07-23  30.675842  4.573082  2.443702  29.769001
307 2022-11-04  34.717156  5.122343  2.460136  32.183312


3. Tạo dữ liệu mô phỏng dự báo lũ lụt sông Mê Kông

In [ ]:
np.random.seed(42)
n = 2000

df_flood = pd.DataFrame({
    'rainfall_mm':              np.random.exponential(10, n), 
    'river_level_m':            np.random.normal(3.5, 1.5, n),
    'soil_moisture':            np.random.uniform(0.2, 0.9, n),
    'elevation_m':              np.random.uniform(0.5, 15, n),
    'slope_deg':                np.random.uniform(0, 8, n),
    'upstream_flow_m3s':        np.random.exponential(50, n),
    'rainfall_7day_cumulative': np.random.exponential(45, n), 
    'ndvi':                     np.random.uniform(0.1, 0.8, n),
})

flood_score = (
    0.40 * (df_flood['rainfall_mm'] > 15).astype(int) +     
    0.30 * (df_flood['river_level_m'] > 4.5).astype(int) +  
    0.30 * (df_flood['soil_moisture'] > 0.65).astype(int)    
)
df_flood['flood_occurred'] = (
    flood_score + np.random.normal(0, 0.15, n) > 0.45
).astype(int)

# Kiểm tra tỷ lệ
print(f"Tỷ lệ lũ: {df_flood['flood_occurred'].mean():.1%}")
print(df_flood['flood_occurred'].value_counts())

df_flood.to_csv("data_set/flood_mekong.csv", index=False)

Tỷ lệ lũ: 26.9%
flood_occurred
0    1462
1     538
Name: count, dtype: int64


In [25]:
pip install rasterio

   ---------------------------------------- 0.0/30.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/30.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/30.1 MB ? eta -:--:--
    --------------------------------------- 0.5/30.1 MB 1.3 MB/s eta 0:00:24
   - -------------------------------------- 0.8/30.1 MB 1.3 MB/s eta 0:00:23
   - -------------------------------------- 1.0/30.1 MB 1.4 MB/s eta 0:00:22
   -- ------------------------------------- 1.6/30.1 MB 1.5 MB/s eta 0:00:19
   -- ------------------------------------- 1.8/30.1 MB 1.5 MB/s eta 0:00:19
   --- ------------------------------------ 2.4/30.1 MB 1.6 MB/s eta 0:00:18
   --- ------------------------------------ 2.9/30.1 MB 1.8 MB/s eta 0:00:16
   ---- ----------------------------------- 3.4/30.1 MB 1.9 MB/s eta 0:00:14
   ----- ---------------------------------- 4.2/30.1 MB 2.1 MB/s eta 0:00:13
   ------ --------------------------------- 5.0/30.1 MB 2.3 MB/s eta 0:00:12
   ------- ---------


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


4. Tạo dữ liệu mô phỏng ảnh vệ tinh Sentinel-2 (NDVI – giám sát rừng)

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds
import os

np.random.seed(42)
H, W = 500, 500

transform = from_bounds(102, 10, 110, 25, W, H)
profile   = {"driver":"GTiff","dtype":"float32",
             "width":W,"height":H,"count":1,"crs":"EPSG:4326",
             "transform":transform}

pixel_ha = 1.0

# 2020 
nir_2020 = np.random.uniform(0.20, 0.40, (H, W)).astype("float32")
red_2020 = np.random.uniform(0.10, 0.20, (H, W)).astype("float32")

nir_2020[:330, :330] = np.random.uniform(0.75, 0.90, (330, 330)).astype("float32")
red_2020[:330, :330] = np.random.uniform(0.03, 0.07, (330, 330)).astype("float32")

# 2023: Mất rừng
nir_2023 = nir_2020.copy()
red_2023 = red_2020.copy()

nir_2023[30:55,  20:56]  -= 0.65   
nir_2023[140:158, 30:60] -= 0.60   
nir_2023[210:228, 50:80] -= 0.55  

red_2023[30:80,   20:90]  += 0.20
red_2023[140:175, 30:85]  += 0.15
red_2023[210:245, 50:100] += 0.12

nir_2023 = np.clip(nir_2023, 0, 1).astype("float32")
red_2023 = np.clip(red_2023, 0, 1).astype("float32")

# Kiểm tra
ndvi_2020c = (nir_2020 - red_2020) / (nir_2020 + red_2020 + 1e-8)
ndvi_2023c = (nir_2023 - red_2023) / (nir_2023 + red_2023 + 1e-8)
delta        = ndvi_2020c - ndvi_2023c
forest_mask  = ndvi_2020c > 0.5
loss_mask    = forest_mask & (delta > 0.3)

total_forest_ha = forest_mask.sum() * pixel_ha  
area_lost_ha    = loss_mask.sum()   * pixel_ha  
loss_rate_pct   = loss_mask.sum()   / forest_mask.sum() * 100

print(f"NDVI 2020 vùng rừng mean : {ndvi_2020c[:330,:330].mean():.3f}")
print(f"Tổng pixel rừng 2020     : {forest_mask.sum():,}")
print(f"Pixel mất rừng           : {loss_mask.sum():,}")
print(f"Diện tích rừng (2020)    : {total_forest_ha:,.0f} ha")
print(f"Diện tích mất  (3 năm)   : {area_lost_ha:,.0f} ha")
print(f"Tỷ lệ mất rừng           : {loss_rate_pct:.2f}%")

os.makedirs("data_set", exist_ok=True)
for fname, arr in [("s2_2020_B08.tif", nir_2020),
                   ("s2_2020_B04.tif", red_2020),
                   ("s2_2023_B08.tif", nir_2023),
                   ("s2_2023_B04.tif", red_2023)]:
    with rasterio.open(f"data_set/{fname}", "w", **profile) as dst:
        dst.write(arr, 1)
    print(f"✓ {fname}")

NDVI 2020 vùng rừng mean : 0.886
Tổng pixel rừng 2020     : 120,658
Pixel mất rừng           : 4,631
Diện tích rừng (2020)    : 120,658 ha
Diện tích mất  (3 năm)   : 4,631 ha
Tỷ lệ mất rừng           : 3.84%
✓ s2_2020_B08.tif
✓ s2_2020_B04.tif
✓ s2_2023_B08.tif
✓ s2_2023_B04.tif


5. Tạo dữ liệu mô phỏng phân loại đất đa lớp 

In [12]:
land_classes = ['Lúa','Hoa màu','Cây lâu năm',
                'Đất thoái hóa','Mặt nước','Đô thị']
feature_names = ['B2_Blue','B3_Green','B4_Red','B8_NIR',
                 'B11_SWIR1','B12_SWIR2',
                 'NDVI','NDWI','EVI','SAVI',
                 'texture_contrast','texture_entropy']
class_means = {
    'Lúa':           [0.05,0.09,0.07,0.45,0.20,0.10,0.65,0.30,0.45,0.50,12,2.1],
    'Hoa màu':       [0.06,0.10,0.08,0.38,0.22,0.12,0.55,0.10,0.38,0.43,14,2.3],
    'Cây lâu năm':   [0.04,0.08,0.06,0.55,0.18,0.09,0.72,0.05,0.55,0.60,10,1.9],
    'Đất thoái hóa': [0.12,0.14,0.15,0.22,0.30,0.22,0.18,-0.1,0.15,0.20,20,2.8],
    'Mặt nước':      [0.03,0.05,0.04,0.08,0.05,0.03,0.05,0.70,0.04,0.05, 5,1.5],
    'Đô thị':        [0.10,0.12,0.13,0.20,0.28,0.20,0.10,-0.2,0.08,0.12,25,3.2],
}
np.random.seed(42)
X_list, y_list = [], []
for cls in land_classes:
    samples = np.random.normal(
        loc=class_means[cls],
        scale=[0.05]*6 + [0.10]*4 + [5, 0.5],
        size=(200, 12)
    )
    X_list.append(samples)
    y_list.extend([cls]*200)

df_land = pd.DataFrame(np.vstack(X_list), columns=feature_names)
df_land['land_class'] = y_list
df_land.to_csv("data_set/sentinel2_land_classification.csv", index=False)
print(df_land.head())

    B2_Blue  B3_Green    B4_Red    B8_NIR  B11_SWIR1  B12_SWIR2      NDVI  \
0  0.074836  0.083087  0.102384  0.526151   0.188292   0.088293  0.807921   
1  0.062098 -0.005664 -0.016246  0.421886   0.149358   0.115712  0.559198   
2  0.022781  0.095546  0.012450  0.468785   0.169968   0.085415  0.589829   
3  0.060443 -0.007984  0.003591  0.459843   0.236923   0.108568  0.638435   
4  0.067181  0.001848  0.086204  0.430746   0.166154   0.130584  0.753100   

       NDWI       EVI      SAVI  texture_contrast  texture_entropy land_class  
0  0.376743  0.403053  0.554256          9.682912         1.867135        Lúa  
1  0.158770  0.596565  0.477422         12.337641         1.387626        Lúa  
2  0.485228  0.448650  0.394229         16.112725         1.489578        Lúa  
3  0.269890  0.302148  0.428016          9.696806         2.628561        Lúa  
4  0.393128  0.366078  0.469079         13.656317         2.587773        Lúa  


6. Tạo dữ liệu mô phỏng nhiệt độ bề mặt biển (SST – giám sát san hô)

In [ ]:
n = 365
dates = pd.date_range('2023-01-01', periods=n)
t = np.linspace(0, 4*np.pi, n)
seasonal = 2 * np.sin(t - np.pi/2)
elnino = np.zeros(n)
elnino[70:170] = 1.1 * np.sin(np.linspace(0, np.pi, 100))
sst = 28.0 + seasonal + elnino + np.random.normal(0, 0.3, n)

df_sst = pd.DataFrame({'date': dates, 'SST_celsius': sst})
df_sst.to_csv("data_set/sst_vietnam_2023.csv", index=False)
print(df_sst.head())